# Hugging Face builder test

In [4]:
# Get the current script directory, from the notebook
import os
notebook_dir = os.getcwd()
print("Notebook directory:", notebook_dir)

model_filename = "v6-7B"
model_file = os.path.join(notebook_dir, ".model", f"{model_filename}.pth")
print("Model file path:", model_file)

# Check if the model file exists
if os.path.isfile(model_file) is False:
    raise Exception("Model file does not exist")

# Get the project directory two levels up
project_dir = os.path.dirname(os.path.dirname(notebook_dir))
print("Project directory:", project_dir)

# Output build directory
output_dir = os.path.join(notebook_dir, f".hf_build/{model_filename}/")
print("Output directory:", output_dir)

Notebook directory: /home/recursal/rwkv-prj/RWKV-block/test/v6_finch
Model file path: /home/recursal/rwkv-prj/RWKV-block/test/v6_finch/.model/v6-7B.pth
Project directory: /home/recursal/rwkv-prj/RWKV-block
Output directory: /home/recursal/rwkv-prj/RWKV-block/test/v6_finch/.hf_build/v6-7B/


In [ ]:
# Empty the output directory, if it exists
if os.path.isdir(output_dir):
    import shutil
    print("Removing existing output directory")
    shutil.rmtree(output_dir)
    
# Run the hf_builder.py FULL BUILD
!python3 "$project_dir/hf_builder/hf_builder.py" --model_class "v6_finch" "$model_file" "$output_dir"

# # Run, and update only the model code (useful while debugging)
# !python3 "$project_dir/hf_builder/hf_builder.py" --model-code-only --model_class "v6_finch" "$model_file" "$output_dir"

-----------------------------
Converting RWKV model to HuggingFace format...
Model Class     : v6_finch
Output Directory: /home/recursal/rwkv-prj/RWKV-block/test/v6_finch/.hf_build/v6-7B/
Model code only mode - skipping tokenizer and weights
-----------------------------
Building rwkv_block into HF code ...
Saving model code files ...
-----------------------------
Successfully copied model code files
-----------------------------


# Basic HELLO WORLD

In [3]:
# Run the hello world test
!python3 "$project_dir/test/hf_model_hello.py" --hf_path "$output_dir"


Loading checkpoint shards: 100%|██████████████████| 2/2 [00:01<00:00,  1.50it/s]
Model and tokenizer loaded successfully
Running on device: cuda:0

---------------------------------
Prompt: HELLO WORLD
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Generated text: "
#define HELLO_WORLD_STRING "Hello World"

#define HELLO
---------------------------------

---------------------------------
Prompt: 
In a shocking finding, scientist discovered a herd of dragons living in a remote, previously unexplored valley, in Tibet. Even more surprising to the researchers was the fact that the dragons spoke perfect Chinese.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Generated text: 
When the team arrived in the valley, they were shocked to find the dragons living in a large
---------------------------------


# MMLU validation testing (smaller set)
**(this is not a substitute for lm-eval-harness : the score is counted differently)**

In [4]:
# MMLU tester directory
mmlu_test_dir = os.path.join(project_dir, "test/mmlu")

# Run the test dataset builder, optional:  --use_validation_set
!python3 {mmlu_test_dir}/BuildTestMMLU.py --hf_model "$output_dir" --n_shot 0 --use_validation_set

## Using HF model tokenizer: /home/recursal/rwkv-prj/RWKV-block/test/v6_finch/.hf_build/v6-7B/
## Loading MMLU cached dataset (n_shot=0,tokenizer=world): /home/recursal/rwkv-prj/RWKV-block/test/mmlu/.mmlu_cache/mmlu-val-t_world-n_0-p_0-c_16-r0.pth
## Done: Dataset has been built and cached


In [7]:
# Run the HF based MMLU tester, with the fla kernel
# Batch size of 24, is for a 1B5 model, n_shot 0, with 24GB vram (ie. 4090)
!python3 {mmlu_test_dir}/RunTestMMLU.py "$output_dir" --batch_size 8 --n_shot 0 --use_validation_set --tmix_backend "fla"

------------------------------------------------
## Loading HF model: /home/recursal/rwkv-prj/RWKV-block/test/v6_finch/.hf_build/v6-7B/
Loading checkpoint shards: 100%|██████████████████| 2/2 [00:01<00:00,  1.29it/s]
------------------------------------------------
## Preparing the dataset
## Loading MMLU cached dataset (n_shot=0,tokenizer=world): /home/recursal/rwkv-prj/RWKV-block/test/mmlu/.mmlu_cache/mmlu-val-t_world-n_0-p_0-c_16-r0.pth
## Done: Dataset has been built and cached
------------------------------------------------
## Starting the MMLU test ...
MMLU Subjects:   0%|                                      | 0/1 [00:00<?, ?it/s]### Running MMLU test : all (count=1531, batches=192) ...

Testing all: 100%|████████████████████████████████████████████████| 192/192 [03:10<00:00,  1.32it/s]
                                                                                                    #### all - accuracy=0.3462 , probability=0.2975
MMLU Subjects: 100%|██████████████████████████

In [6]:
# Run the HF based MMLU tester, with the pytorch kernel
# Batch size of 24, is for a 1B5 model, n_shot 0, with 24GB vram (ie. 4090)
# Batch size of 8, is for a 7B model, n_shot 0, with 24GB vram (ie. 4090)
!python3 {mmlu_test_dir}/RunTestMMLU.py "$output_dir" --batch_size 8 --n_shot 0 --use_validation_set --tmix_backend "pytorch"

------------------------------------------------
## Loading HF model: /home/recursal/rwkv-prj/RWKV-block/test/v6_finch/.hf_build/v6-7B/
Loading checkpoint shards: 100%|██████████████████| 2/2 [00:01<00:00,  1.13it/s]
------------------------------------------------
## Preparing the dataset
## Loading MMLU cached dataset (n_shot=0,tokenizer=world): /home/recursal/rwkv-prj/RWKV-block/test/mmlu/.mmlu_cache/mmlu-val-t_world-n_0-p_0-c_16-r0.pth
## Done: Dataset has been built and cached
------------------------------------------------
## Starting the MMLU test ...
MMLU Subjects:   0%|                                      | 0/1 [00:00<?, ?it/s]### Running MMLU test : all (count=1531, batches=192) ...

Testing all: 100%|████████████████████████████████████████████████| 192/192 [04:10<00:00,  1.06s/it]
                                                                                                    #### all - accuracy=0.2502 , probability=0.2481
MMLU Subjects: 100%|██████████████████████████

# MMLU testing 
**(this is not a substitute for lm-eval-harness : the score is counted differently)**

In [ ]:
# Run the HF based MMLU tester, with the cuda kernel
# Batch size of 24, is for a 1B5 model, n_shot 0, with 24GB vram (ie. 4090)
!python3 {mmlu_test_dir}/RunTestMMLU.py "$output_dir" --batch_size 8 --n_shot 0 --tmix_backend "fla"

In [ ]:
# Run the HF based MMLU tester, with the triton kernel (modified)
# Batch size of 24, is for a 1B5 model, n_shot 0, with 24GB vram (ie. 4090)
!python3 {mmlu_test_dir}/RunTestMMLU.py "$output_dir" --batch_size 8 --n_shot 0 --tmix_backend "pytorch"

# LM Eval harness testing
The real MMLU test

In [3]:
# Test the base model
!NCCL_IB_DISABLE=1 NCCL_P2P_DISABLE=1 lm_eval --model hf \
    --model_args pretrained="$output_dir",dtype="bfloat16",trust_remote_code=True,tmix_backend="fla" \
    --tasks mmlu \
    --device "cuda:7" \
    --batch_size 8 # This is adjusted for a 24GB system

2025-03-16:06:57:05,422 INFO     [lm_eval.__main__:379] Selected Tasks: ['mmlu']
2025-03-16:06:57:05,425 INFO     [lm_eval.evaluator:177] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-03-16:06:57:05,425 INFO     [lm_eval.evaluator:214] Initializing hf model, with arguments: {'pretrained': '/home/recursal/rwkv-prj/RWKV-block/test/v6_finch/.hf_build/v6-7B/', 'dtype': 'bfloat16', 'trust_remote_code': True, 'tmix_backend': 'fla'}
2025-03-16:06:57:05,659 INFO     [lm_eval.models.huggingface:136] Using device 'cuda:7'
2025-03-16:06:57:06,391 INFO     [lm_eval.models.huggingface:504] Model type cannot be determined. Using default model type 'causal'
2025-03-16:06:57:08,781 INFO     [lm_eval.models.huggingface:377] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:7'}
Loading checkpoint shards: 100%|██████████████████| 2/2 [00:07<00:00,  3.59s/it]
2025-03-16:06:57: